# Module 3 Composite Resilience Index, Real Data

See `earth-intelligence-toolkit/daear_toolkit/_live.py`'s module docstring
for a per-source confidence note before relying on this under deadline
pressure Sentinel-2/DEM/FIRMS are high-confidence stable APIs; MTBS and
SSURGO have medium-confidence caveats spelled out there.

## Setup (run once)
```bash
pip install -e ../../earth-intelligence-toolkit
pip install -e "../../earth-intelligence-toolkit[live]"
export FIRMS_MAP_KEY=your_key_here   # free: https://firms.modaps.eosdis.nasa.gov/api/map_key/
```


In [ ]:
import matplotlib.pyplot as plt
import daear_toolkit as dt
from daear_toolkit import indicators, viz
from daear_toolkit._live import sentinel2_scene, copernicus_dem, burn_severity_from_sentinel2

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox
print(REGION.name)
print(f"Fire: {REGION.fire_name}, {REGION.fire_start} – {REGION.fire_contained}, "
      f"{REGION.fire_acres:,} acres (verified figure)")

## Step 1: Burn severity via dNBR (no MTBS dependency)

This is the high-confidence path: two real Sentinel-2 scenes, real dNBR.
If you'd rather use MTBS's own severity product, see
`mtbs_burn_severity_from_direct_download()` in `_live.py` instead —
you'll need to find the direct GeoTIFF URL for Cameron Peak at
https://mtbs.gov/direct-download first.

In [ ]:
# Pre-fire: a clear-sky scene from shortly before the Aug 13, 2020 ignition.
# Post-fire: roughly a year later, once the burn scar has stabilized enough
# for a clean dNBR read (avoids transient ash/smoke artifacts right after
# containment). Adjust dates if cloud cover forces a different pick --
# sentinel2_scene() will raise a clear error naming the search window if
# no scene is found.
burn = burn_severity_from_sentinel2(BBOX, pre_fire_date="2020-06-15", post_fire_date="2021-07-15")

fig, ax = plt.subplots(figsize=(6, 5))
viz.plot_raster(burn, title="Real dNBR — Cameron Peak Fire", ax=ax, cmap=viz.SEVERITY_CMAP)
plt.tight_layout()
plt.savefig("../outputs/03_LIVE_burn_severity.png", dpi=150)
plt.show()

## Step 2: real terrain

In [ ]:
terrain = copernicus_dem(BBOX)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_raster(terrain["elevation_m"], title="Real elevation (Copernicus 30m DEM)", ax=axes[0], cmap="terrain")
viz.plot_raster(terrain["slope_deg"], title="Real slope (deg)", ax=axes[1], cmap=viz.SEVERITY_CMAP)
plt.tight_layout()
plt.savefig("../outputs/03_LIVE_terrain.png", dpi=150)
plt.show()

## Step 3: soil — the one piece that needs an extra step

SSURGO is map-unit polygon data, not a regular grid, so it doesn't drop
directly into `indicators.soil_vulnerability()` the way the synthetic demo
data does. This cell fetches the real attribute table; rasterizing it onto
the same grid as `burn`/`terrain` (e.g. with `geopandas` + `rasterio.features.rasterize`)
is the follow-up step, not included here since it needs real map unit
geometries alongside this attribute table.

In [ ]:
from daear_toolkit._live import ssurgo_soil_properties

soil_table = ssurgo_soil_properties(BBOX)
soil_table.head()

# TODO once you have this running: rasterize organic_matter_pct and
# sand_pct onto burn's lat/lon grid using the map unit geometries (fetch
# via SDA's spatial query or SSURGO's official geometry download), then
# call indicators.soil_vulnerability() as in the synthetic-data notebook.

## Step 4: composite index (once soil is rasterized)

With `burn`, `terrain`, and a rasterized soil layer on the same grid, the
rest is identical to the synthetic-data notebook
(`03_composite_resilience_index.ipynb`) — same
`indicators.erosion_susceptibility()` and
`indicators.watershed_resilience_index()` calls, now running on real data:

```python
erosion = indicators.erosion_susceptibility(terrain["slope_deg"], burn, soil_runoff_potential_rasterized)
composite = indicators.watershed_resilience_index(erosion, soil_vulnerability_rasterized)
```

## Summary

Three of four inputs (burn severity, terrain, and — via FIRMS — active-fire
context) are fully real and ready to run as soon as this executes somewhere
with network access. Soil is real data too, just one rasterization step
away from dropping into the same composite-index formula already proven
out on synthetic data in `03_composite_resilience_index.ipynb`.
